# Brain Tumor Detection Pipeline — Demo

Screenshots below are from the actual running app (`README.md` → *Running the
app*), analyzing a real MRI scan from the held-out test set. Nothing here is
mocked — this is what the tool produces end-to-end.


## Analysis

### 1. Three-model ensemble agreement

Every scan is classified independently by three CNNs (ConvNeXt-Tiny,
EfficientNet-B3, ResNet-50), each with test-time augmentation. All three
have to actually agree before the app presents a confident result — this
run, all three unanimously predict **Meningioma**. The best-confidence model
(ConvNeXt-Tiny here) also reports MC-Dropout uncertainty from 20 stochastic
passes.

![Model comparison — 3/3 unanimous](assets/analysis/01_model_comparison.png)


### 2. Trust verdict — flagged for review

Even with a confident, unanimous prediction, the app runs independent
consistency checks (uncertainty threshold, focus-crop re-classification,
out-of-distribution score) and will flag a scan for expert review if any of
them trip — rather than only ever showing green when the classifier
sounds confident.

![Caution banner — multiple flags](assets/analysis/02_caution_banner.png)


### 3. Malignancy assessment

Combines the classifier's output with an independently computed tumor size
(YOLO detection + MobileSAM pixel-tight segmentation) and anatomical
location into a 0–10 malignancy score. Two size estimates are shown
side-by-side — the pixel-based pipeline's and MedGemma's own independent
visual estimate — so a disagreement between them is visible rather than
silently resolved.

![Malignancy assessment with tumor bounding box](assets/analysis/03_malignancy_assessment.png)


### 4. Symptom-aware scoring + clinical context

Users can optionally tap any symptoms they've noticed, which adjusts the
score transparently (shown as a visible bonus, not hidden inside a black
box). Below that, reference clinical context for the predicted tumor type —
static medical knowledge, not LLM-generated, so it's consistent every time.

![Symptoms and clinical context](assets/analysis/04_symptoms_clinical_context.png)


### 5. Doctor-visit prep + multi-color anatomical views

A ready-to-use list of follow-up questions for the specific tumor type, plus
the same MRI slice re-rendered in several color maps (HOT, JET, BONE,
VIRIDIS) — different colormaps make different tissue contrasts visible that
a single grayscale view can hide.

![Questions for your doctor + anatomy color schemes](assets/analysis/05_questions_anatomy.png)


### 6. Hierarchical 4-level explainability

Four increasingly specific heat-maps answering four different questions
about the same prediction — Grad-CAM (*is there a tumor?*), Grad-CAM++
(*what type?*), LayerCAM block4 (*where exactly?*), and fused LayerCAM
(*how does the model's reasoning combine across layers?*). All four
consistently highlight the same lesion, which is itself a sanity check —
if they didn't agree with each other, that would suggest the classifier's
attention isn't actually anchored on the tumor.

![Hierarchical 4-level XAI](assets/analysis/06_hierarchical_xai.png)


### 7. MedGemma diagnostic report

The last step: MedGemma 1.5 4B writes a patient-facing narrative from the
full structured pipeline output (prediction, uncertainty, malignancy,
region) — five short sections, plain language, no jargon. This ran on CPU
and took a few minutes; MedGemma occasionally needs a retry here (see the
README troubleshooting section) since it sometimes emits internal reasoning
instead of the formatted report on the first attempt.

![MedGemma diagnostic report](assets/analysis/07_medgemma_report.png)
